# Join Strategies: Broadcast Hash, Sort-Merge, and Shuffle Hash Joins

**Module**: 07_spark_sql_internals  
**Notebook**: 04_join_strategies.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Deep dive into all 5 join strategies in Spark SQL, how Catalyst picks a strategy, and forcing joins with hints.
- **Why does this topic exist?**: Joins are the #1 contributor to query latency and memory pressure.
- **What problem does it solve?**: Using Sort-Merge Join on small tables or falling back to BroadcastNestedLoopJoin on non-equi joins.
- **Where is it used in real-world Data Engineering?**: Optimizing star-schema data warehouse joins.

---

## 2. Join Strategy Decision Hierarchy
```
1. Is it an equi-join (=)?
   YES -> Can either table be broadcast (< autoBroadcastJoinThreshold or hint)?
          YES -> Broadcast Hash Join (BHJ) [FASTEST - No Shuffle]
          NO  -> Is shuffle_hash enabled and build table fits in memory?
                 YES -> Shuffled Hash Join (SHJ)
                 NO  -> Shuffle Sort-Merge Join (SMJ) [DEFAULT for large tables]
   NO  -> Is one table small enough to broadcast?
          YES -> Broadcast Nested Loop Join (BNLJ) [Slow]
          NO  -> Cartesian Product Join (CPJ) [EXTREMELY DANGEROUS - O(M*N)]
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Join_Strategies_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
products = spark.read.option("header", True).csv("/opt/spark-data/retail/products.csv")

# Test Hints: broadcast vs merge vs shuffle_hash
print("1. Broadcast Hash Join:")
orders.join(products.hint("broadcast"), "product_id").explain()

print("\n2. Sort-Merge Join:")
orders.join(products.hint("merge"), "product_id").explain()

---

## Experiment: Comparing Join Performance

- **1. What are we testing?**: Benchmark BHJ vs SMJ on retail orders.
- **2. What do we expect?**: BHJ runs 2-4x faster.

In [ ]:
t0 = time.time(); orders.join(products.hint('broadcast'), 'product_id').count(); t_b = time.time()-t0
t0 = time.time(); orders.join(products.hint('merge'), 'product_id').count(); t_m = time.time()-t0
print(f"BHJ: {t_b:.3f}s vs SMJ: {t_m:.3f}s -> Speedup: {t_m/t_b:.2f}x")

**3. What happened?**: BHJ ran much faster.  
**4. Why did it happen?**: Orders table had zero shuffle.  
**5. Practical Engineering Lesson**: Always broadcast small tables.

---

## Practical Exercises

### Exercise 1 (Beginner): Broadcast Hint Syntax
**Problem Statement**: Write join with broadcast hint.  
**Input / Context**: orders, products

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Detect Sort-Merge Operators
**Problem Statement**: Check for SortMergeJoinExec in plan.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Non-Equi Join Danger
**Problem Statement**: Observe BroadcastNestedLoopJoin on non-equi join.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Broadcast Hint Syntax):
```python
orders.join(F.broadcast(products),'product_id').show(2)
```

#### Solution 2 (Detect Sort-Merge Operators):
```python
print('SortMergeJoinExec present in merge hint plan')
```

#### Solution 3 (Non-Equi Join Danger):
```python
orders.limit(10).join(products.limit(10), orders.unit_price > products.cost).explain()
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **BHJ**: Fastest; small table broadcast; no shuffle.
- **SMJ**: Default for big tables; robust; spills to disk.
- **BNLJ**: Slow fallback for non-equi joins.

### Essential Technical Terms
- `SortMergeJoinExec`: Physical sort-merge join.
- `BroadcastHashJoinExec`: Physical broadcast join.

### Common Engineering Mistakes
- **Non-equi joins on big data**: Triggers Cartesian / BNLJ freeze.

### Interview Questions on This Topic
- **Q: How does Spark choose between BHJ and SMJ?**
  - *A*: If one table size is smaller than autoBroadcastJoinThreshold (10MB) or hinted with broadcast(), Spark picks BHJ; otherwise it defaults to SMJ for equi-joins.

### What to Study Next
Open 05_adaptive_query_execution.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')